In [3]:
#1.CARGA DEL ARCHIVO:
# Con este código elegimos el archivo o los archivos del escritorio que deseamos cargar y leer el contenido:
from google.colab import files
#Seleccionar y guardar el archivo de Excel
uploaded = files.upload()

Saving f1_2016.xlsx to f1_2016.xlsx


In [4]:
# Importamos la librería que nos permitirá trabajar con achivos csv o excel
import pandas as pd

# Con este código asignamos una variable al archivo importado y se lee el archivo Excel
df_f1 = pd.read_excel('f1_2016.xlsx')

# Mostrar el dataframe, en este caso hemos mostrado todos los datos de la tabla para posteriormente filtrar solo los datos de los pilotos seleccionados:
print(df_f1)

           Piloto  AUS  BHR  CHN  RUS  ESP  MCO  CAN  EUR  AUT  ...  DEU  BEL  \
0      N. ROSBERG   25   25   25   25    0    6   10   25   12  ...   12   25   
1     L. HAMILTON   18   15    6   18    0   25   25   10   25  ...   25   15   
2    D. RICCIARDO   12   12   12    0   12   18    6    6   10  ...   18   18   
3       S. VETTEL   15    0   18    0   15   12   18   18    0  ...   10    8   
4   M. VERSTAPPEN    1    8    4    0   25    0   12    4   18  ...   15    0   
5    K. RAIKKONEN    0   18   10   15   18    0    8   12   15  ...    8    2   
6        S. PEREZ    0    0    0    2    6   15    1   15    0  ...    1   10   
7       V. BOTTAS    4    2    1   12   10    0   15    8    2  ...    2    4   
8   N. HULKENBERG    6    0    0    0    0    8    4    2    0  ...    6   12   
9       F. ALONSO    0    0    0    8    0   10    0    0    0  ...    0    6   
10       F. MASSA   10    4    8   10    4    1    0    1    0  ...    0    1   
11       C. SAINZ    2    0 

In [5]:
#2. FILTRADO DE DATOS DE LOS PILOTOS SELECCIONADOS:
# Definimos la lista con los pilotos exactos a filtrar
pilotos_deseados = ["S. VETTEL", "S. PEREZ"]

# Creamos una lista vacía para almacenar temporalmente las filas completas de los pilotos encontrados
filas_filtradas = []

# Recorremos el DataFrame fila por fila
for i in range(len(df_f1)):
    piloto = df_f1.loc[i, "Piloto"]

    # Verificamos si el piloto de la fila actual se encuentra dentro de nuestra lista deseada
    if piloto in pilotos_deseados:
        # Extraemos y guardamos toda la fila correspondiente al piloto en la lista
        filas_filtradas.append(df_f1.iloc[i])

# Creamos el nuevo DataFrame con todas las columnas de los pilotos seleccionados
df_filtrado = pd.DataFrame(filas_filtradas)
print(df_filtrado)

      Piloto  AUS  BHR  CHN  RUS  ESP  MCO  CAN  EUR  AUT  ...  DEU  BEL  ITA  \
3  S. VETTEL   15    0   18    0   15   12   18   18    0  ...   10    8   15   
6   S. PEREZ    0    0    0    2    6   15    1   15    0  ...    1   10    4   

   SGP  MYS  JPN  USA  MEX  BRA  ABD  
3   10    0   12   12   10   10   15  
6    4    8    6    4    1   12    4  

[2 rows x 22 columns]


In [6]:
#2.CALCULAMOS LA SUMA DE PUNTAJE POR PILOTO
# Definimos la lista con los nombres exactos de los pilotos que deseamos consultar
pilotos_buscados = ["S. VETTEL", "S. PEREZ"]
# Recorremos cada piloto de la lista de búsqueda para filtrar sus registros y sumar sus puntos:
for busqueda in pilotos_buscados:
    resultado = df_f1.loc[df_f1["Piloto"].str.lower() == busqueda.lower()]
# Verificamos si el resultado no está vacío antes de procesar los datos:
    if not resultado.empty:
        # Sumamos todas las columnas de puntaje de las carreras
        puntos = pd.to_numeric(resultado.iloc[0, 1:], errors="coerce").sum()
        print("Total de puntos de", resultado.iloc[0, 0], round(puntos))# Imprimimos en pantalla el resultado redondeado del puntaje total por piloto
    else:
        print("No se encontró el piloto:", busqueda) #Mensaje de advertencia en caso de que no se encuentre el piloto en el DataFrame

Total de puntos de S. VETTEL 212
Total de puntos de S. PEREZ 101


In [11]:
 #4. CREACIÓN DE UN DICCIONARIO:
# Creamos un diccionario que contenga la información de la escudería, enlace de la fotografía, latitud y longitud dentro del string de "coordenadas_lugares" de los dos pilotos seleccionados:
diccionario_pilotos = {
    "S. VETTEL": {
        "Escudería": "Scuderia Ferrari",  # Equipo temporada 2016
        "Enlace de la fotografía": "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/af/Sebastian_Vettel_at_the_2026_Italian_Grand_Prix.jpg/250px-Sebastian_Vettel_at_the_2026_Italian_Grand_Prix.jpg?utm_source=es.wikipedia.org&utm_campaign=parser&utm_content=thumbnail",
        "coordenadas_lugares": [49.643056, 8.638889], "Puntaje": 212              # Nacionalidad: Heppenheim, Alemania
        },
    "S. PEREZ": {
        "Escudería": "Sahara Force India",  # Equipo temporada 2016
        "Enlace de la fotografía": "https://upload.wikimedia.org/wikipedia/commons/7/7a/Sergio_P%C3%A9rez_2019_%28cropped%29.jpg?utm_source=es.wikipedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
        "coordenadas_lugares":[20.676667, -103.3475], "Puntaje": 101             # Nacionalidad: Guadalajara, México
    }
}

In [17]:
#5. CREACIÓN DE UN MAPA FOLIUM:
#CREAMOS UN MAPA BASE

# Importamos la librería folium que sirve para crear mapas interactivos
import folium

# Creamos el mapa con OpenStreetMap y establecemos las coordenadas centrales
mapa_1 = folium.Map(location=[20, 0], zoom_start=4, tiles=None, control_scale=True)#establecemos las coordenadas de referencia y el número del zoom que permite visualizar la ubicación ampliamente
# GENERAMOS LAS LICENCIAS PARA SU ADECUADA VISUALIZACIÓN:
folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        'OpenStreetMap contributors</a> '
        '| Datos bajo licencia '
        '<a href="https://opendatacommons.org/licenses/odbl/">ODbL</a>'
    ),
    name="OpenStreetMap",
    referrer_policy="strict-origin"
).add_to(mapa_1)
#PERSONALZIAR EL ÍCONO Y EL COLOR DE LOS MARCADORES:
# Recorremos el diccionario de pilotos para agregar un marcador por cada uno utilizando sus coordenadas
for piloto, datos in diccionario_pilotos.items():
    # Obtenemos las coordenadas desde las claves 'Latitud' y 'Longitud'
    coords = [datos["Latitud"], datos["Longitud"]]
    folium.Marker(
        location=coords,         # Coordenadas del lugar
        popup=piloto,            # Texto que aparece al hacer clic en el marcador (nombre del piloto)
        icon=folium.Icon(color='orange', icon="car", prefix="fa")  # Estilo del ícono del marcador que es un carro que se asocia a las carreras y de color naranja
    ).add_to(mapa_1)  # Agregamos el marcador al mapa

# Mostramos el mapa interactivo
mapa_1

In [20]:
# CREACIÓN DEL MAPA CON UNA VENTANA EMERGENTE
# Creamos el diccionario que incluye toda la información de los pilotos, incluyendo el puntaje total para su uso posterior
diccionario_pilotos = {
    "S. VETTEL": {
        "Nacionalidad": "Alemania",
        "Escudería": "Scuderia Ferrari",
        "Link": "https://thumb.wikimedia.org/wikipedia/commons/thumb/a/af/Sebastian_Vettel_at_the_2026_Italian_Grand_Prix.jpg/250px-Sebastian_Vettel_at_the_2026_Italian_Grand_Prix.jpg?utm_source=es.wikipedia.org&utm_campaign=parser&utm_content=thumbnail",
        "Latitud": 49.643056,
        "Longitud": 8.638889,
        "Puntaje": 212
    },
    "S. PEREZ": {
        "Nacionalidad": "México",
        "Escudería": "Sahara Force India",
        "Link": "https://upload.wikimedia.org/wikipedia/commons/7/7a/Sergio_P%C3%A9rez_2019_%28cropped%29.jpg?utm_source=es.wikipedia.org&utm_campaign=imageinfo&utm_content=thumbnail_unscaled",
        "Latitud": 20.676667,
        "Longitud": -103.3475,
        "Puntaje": 101
    }
}

# Importamos folium y creamos de nuevo el mapa para añadir las ventanas emergentes
import folium
mapa_2 = folium.Map(location=[20, 0], zoom_start=2, tiles=None, control_scale=True)

# Agregamos OpenStreetMap
folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap contributors</a>',
    name="OpenStreetMap",
    referrer_policy="strict-origin"
).add_to(mapa_2)

# Convertimos el diccionario a un DataFrame de pandas para definir 'df_pilotos_mapa'
df_pilotos_mapa = pd.DataFrame.from_dict(diccionario_pilotos, orient='index').reset_index().rename(columns={'index': 'Piloto'})

# Agrupamos los datos por nacionalidad
agrupado = df_pilotos_mapa.groupby("Nacionalidad")

# Recorremos cada grupo
for pais, grupo in agrupado:

    lat = grupo["Latitud"].iloc[0]
    lon = grupo["Longitud"].iloc[0]

    if pd.notna(lat) and pd.notna(lon):

        contenido = ""

        # Recorremos los pilotos de cada grupo usando iterrows
        for elemento, fila in grupo.iterrows():

            piloto = fila["Piloto"]
            escuderia = fila["Escudería"]
            imagen = fila["Link"]
            puntos = fila["Puntaje"]  # Obtenemos el puntaje directo del diccionario ya creado anteriormente

            # Agregamos la imagen si existe un enlace válido
            if pd.notna(imagen) and imagen != "":
                contenido += f"""
                <div style="text-align:center; margin-bottom:15px;">

                    <img src="{imagen}"
                         width="100"
                         height="100"
                         style="object-fit:cover; border-radius:50%;">
                    <h1><b><i>{piloto}</i></b></h1><br>
                    <b><span>{escuderia}</span><br></b>
                    <span><b>Puntaje total:</b> {puntos} pts</span>

                </div>
                """

            else:
                contenido += f"""
                <div style="text-align:center; margin-bottom:15px;">
                    <h1><b><i>{piloto}</i></b></h1><br>
                    <b><span>{escuderia}</span></b><br>
                    <span><b>Puntaje total:</b> {puntos} pts</span>
                </div>
                """

        # Agregamos el marcador
        folium.Marker(
            location=[lat, lon],
            popup=folium.Popup(contenido, max_width=350),
            tooltip=pais,
            icon=folium.Icon(color="orange", icon="car", prefix='fa')
        ).add_to(mapa_2)
mapa_2.save("mapa_base.html")
# Mostramos el mapa interactivo final
mapa_2

## **JUSTIFICACIÓN**
La estructuración de la ventana emergente se diseñó bajo un enfoque visual limpio, jerárquico y centrado, usando etiquetas fundamentales para organizar la información de los pilotos como la etiqueta i que se implementó para colocar los nombres de ambos pilotos en cursiva, otorgándoles un estilo diferenciado y estético dentro de la tarjeta de presentación, asimismo el h1 para destacar los nombres de los pilotos como un título principal de mayor tamaño, lo cual permite que se resalte de inmediato al abrir la ventama emergente y facilitando la identificación rápida del perfil del piloto. Además, la etiqueta b que se aplicó tanto para resaltar el nombre de la escudería de cada piloto, dándolo peso visual frente al resto de la información complementaria, y destacar la palabra "puntaje total" que permite que el lector localice de manera inmediata la cifra exacta del total de puntaje obtenido de las carreras por cada piloto, de manera que el valor numérico se mantiene en tipografía estándar para equilibrar la lectura.
Por otro lado, se utilizó un bloque con alineación central (text-align: center) para mantener una simetría estética y equilibrada, evitando que los elementos se dispersen alrededor del marcador geográfico.
Por último, se ubicó primero la fotografía del piloto con bordes redondeados, permitiendo que el usuario identifique visualmente al deportista de manera inmediata antes de leer los datos textuales, luego el nombre del piloto y en la parte de abajo se coloca la información complementaria en líneas independientes correspondiente a la escudería y el puntaje total, permitiendo que se lea la información de manera secuencial.